[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsarti/ik-nlp-tutorials/blob/main/notebooks/W5E_Inseq_Analysis.ipynb)

In [1]:
# Run in Colab to install local packages
!pip install inseq

# Exercise 1: Analyzing language generation models with Inseq 🐛

*Adapted in part from the [Inseq documentation](https://inseq.readthedocs.io/)*

Inseq is a toolkit based on the 🤗 Transformers and [Captum](https://captum.ai/docs/introduction) libraries for interpreting language generation models using feature attribution methods. Inseq allows you to analyze the behavior of a language generation model by computing the importance of each input token for each token in the generated output. The importance can be obtained using approaches based on attention, gradients and more, which we will see in more detail in the final lecture.

Inseq is a relatively new library, and it is still under active development (contributions welcome! 🙂). You can refer to the [Inseq paper](https://arxiv.org/abs/2302.13942) for an overview of the tool and some examples, or [this paper](https://www.semanticscholar.org/paper/Are-Character-level-Translations-Worth-the-Wait-An-Edman-Toral/ed7b51e4a5c4835218f6697b280afb2849211939) for a recent work from our GroNLP group on using Inseq to analyze character-level translation models.

In the following sections two simple use-cases of Inseq are presented.

## Attributing (Un)constrained Machine Translation

In this section we will use Inseq to compute the importance of each input token for each token in the generated output. We will use the [Helsinki-NLP/opus-mt-en-nl](https://huggingface.co/Helsinki-NLP/opus-mt-en-nl) model, which is a pretrained machine translation model from English to Dutch.

In [3]:
import inseq
import warnings

warnings.filterwarnings("ignore")

model = inseq.load_model("Helsinki-NLP/opus-mt-en-nl", "input_x_gradient")
out = model.attribute(
    "Don't get hot-headed mate, it's easy peasy to study models with Inseq!",
    attribute_target=True,
    step_scores=["probability"]
)
out.show()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/316M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/316M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Attributing with input_x_gradient...: 100%|██████████| 20/20 [00:06<00:00,  3.01it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
,,▁Maak,▁je,▁niet,▁zo,▁druk,",",▁het,▁is,▁makkelijk,▁om,▁modellen,▁te,▁bestuderen,▁met,▁In,se,q,!,</s>
0,▁Don,0.045,0.035,0.018,0.022,0.019,0.015,0.02,0.009,0.018,0.003,0.009,0.005,0.014,0.004,0.007,0.005,0.004,0.004,0.011
1,',0.019,0.012,0.011,0.015,0.009,0.01,0.01,0.005,0.014,0.001,0.007,0.003,0.01,0.003,0.004,0.004,0.002,0.002,0.011
2,t,0.031,0.021,0.019,0.016,0.014,0.012,0.015,0.007,0.008,0.001,0.005,0.003,0.007,0.002,0.006,0.003,0.002,0.002,0.007
3,▁get,0.118,0.053,0.035,0.035,0.025,0.018,0.025,0.009,0.013,0.003,0.011,0.006,0.013,0.004,0.009,0.004,0.003,0.004,0.01
4,▁hot,0.066,0.063,0.072,0.077,0.077,0.039,0.025,0.019,0.015,0.004,0.012,0.009,0.01,0.008,0.008,0.007,0.006,0.006,0.01
5,-,0.021,0.017,0.021,0.023,0.023,0.013,0.008,0.006,0.007,0.001,0.005,0.003,0.006,0.003,0.005,0.004,0.005,0.002,0.006
6,headed,0.143,0.123,0.188,0.121,0.173,0.097,0.048,0.038,0.027,0.007,0.027,0.02,0.039,0.024,0.017,0.012,0.009,0.014,0.022
7,▁mate,0.085,0.056,0.06,0.055,0.061,0.059,0.033,0.014,0.017,0.003,0.012,0.008,0.016,0.008,0.008,0.005,0.004,0.006,0.01
8,",",0.02,0.015,0.013,0.013,0.011,0.017,0.017,0.008,0.012,0.001,0.005,0.003,0.007,0.003,0.004,0.003,0.003,0.003,0.01


<details>
    <summary> Observations </summary>

- The model translates idiomatic expressions in a non-literal way. Attribution scores reflect that the model is attributing strong importance to pieces of the idiom when translating (e.g. `_headed`, `_peas`), while also accounting for the prefix when producing an idiomatic translation (e.g. `_Maak`).
- The model gets progressively more confident in its translation as it generates more tokens. The first generated token is very unlikely.
</details>

Let's try now to **constrain** the generation to a more literal translation of the input. Attributing a prespecified output can be intuitively thought as a way to ask a model to justify a possible prediction. Note that this should be done with care, since if the output is very unlikely the results will be very noisy.

In [4]:
import inseq

model = inseq.load_model("Helsinki-NLP/opus-mt-en-nl", "input_x_gradient")
out = model.attribute(
    "Don't get hot-headed mate, it's easy peasy to study models with Inseq!",
    #"Niet heethoofdig worden maatje, het is gemakkelijk peasy om modellen te bestuderen met Inseq!",
    "Niet heethoofdig worden man, het is een makkie om modellen te bestuderen met Inseq!",
    attribute_target=True,
    step_scores=["probability"]
)
out.show()

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Attributing with input_x_gradient...: 100%|██████████| 24/24 [00:07<00:00,  2.91it/s]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
,,▁Niet,▁heet,hoofd,ig,▁worden,▁man,",",▁het,▁is,▁een,▁,mak,kie,▁om,▁modellen,▁te,▁bestuderen,▁met,▁In,se,q,!,</s>
0,▁Don,0.08,0.038,0.01,0.02,0.021,0.021,0.017,0.016,0.013,0.014,0.008,0.008,0.004,0.003,0.009,0.003,0.011,0.003,0.006,0.005,0.003,0.001,0.007
1,',0.031,0.022,0.006,0.015,0.011,0.013,0.013,0.013,0.007,0.009,0.006,0.006,0.002,0.002,0.006,0.002,0.008,0.003,0.004,0.004,0.001,0.001,0.006
2,t,0.045,0.022,0.011,0.014,0.017,0.017,0.011,0.01,0.007,0.007,0.005,0.005,0.003,0.002,0.004,0.003,0.005,0.002,0.004,0.003,0.002,0.001,0.004
3,▁get,0.095,0.04,0.018,0.044,0.048,0.04,0.018,0.02,0.014,0.011,0.008,0.009,0.004,0.003,0.009,0.005,0.011,0.004,0.007,0.004,0.002,0.001,0.005
4,▁hot,0.082,0.09,0.101,0.062,0.05,0.044,0.018,0.022,0.017,0.013,0.01,0.012,0.009,0.008,0.011,0.011,0.009,0.008,0.008,0.007,0.008,0.002,0.007
5,-,0.023,0.023,0.033,0.022,0.013,0.022,0.009,0.008,0.006,0.006,0.005,0.005,0.004,0.003,0.004,0.004,0.005,0.003,0.003,0.004,0.003,0.001,0.003
6,headed,0.126,0.191,0.217,0.162,0.074,0.152,0.063,0.056,0.033,0.027,0.022,0.032,0.032,0.023,0.026,0.04,0.032,0.02,0.023,0.013,0.021,0.005,0.015
7,▁mate,0.08,0.05,0.051,0.132,0.035,0.149,0.075,0.036,0.025,0.016,0.012,0.013,0.019,0.016,0.016,0.017,0.016,0.008,0.008,0.005,0.009,0.002,0.01
8,",",0.025,0.019,0.012,0.027,0.009,0.022,0.012,0.013,0.012,0.009,0.005,0.006,0.003,0.003,0.005,0.003,0.006,0.002,0.003,0.004,0.002,0.001,0.005


### Your turn to comment

Comment on the resulting scores from the constrained, less idiomatic example, putting them in relation to the unconstrained, more idiomatic one. Consider the following aspects, but feel free to explore other examples and add your own observations:
1. How are importance scores distributed on idiomatic and non-idiomatic tokens?
2. What is the difference in probability between the two examples?
3. Do you notice some patterns regarding the low-probability tokens in the second example?
4. When is the target prefix playing a more important role in the generation, according to the attribution scores?

### Comment

In the source saliency map related to the unconstrained (more idiomatic) example, importance scores are generally more concentrated. Specific input tokens strongly contribute to their idiomatic translations: idiomatic expressions such as “hot-headed” and “easy peasy”, for example, are treated as unified semantic units.

In contast, in the constrained (less idiomatic) example, importance scores are more dispersed. When the translation is forced into a more literal form (e.g., “heethoofdig” for “hot-headed”), attribution values are distributed over smaller morphological components rather than concentrated on a single semantic unit. This highlights that the model shows a lower level of confidence in the generation.

There is also a clear difference in token probabilities. In the unconstrained example, probabilities are generally higher and more stable across the sentence. Although the first token (“Maak”) has relatively low probability, subsequent tokens quickly reach moderate to high values (many above 0.5 or even 0.8-0.9). In contrast, in the constrained example, several tokens exhibit extremely low probabilities (many close to 0.01 or lower).

Regarding the role of the target prefix (as seen in the target saliency map), the constrained example shows higher values along the diagonal, suggesting stronger dependence on previously generated tokens. This seems to indicate that, when producing unlikely words, the model relies more heavily on the existing target context to maintain syntactic coherence.

## Contrastive Attribution for Motivating Preferences

In the previous section we used importance scores produced by attributing next token’s probability, which can be seen as answering the question “Which elements of the input are the most relevant to produce the next generation step?”.

However, in many cases we might be more interested in understanding why our model generated its output **rather than another one that we consider to be more likely**. The paper [“Interpreting Language Models with Contrastive Explanations”](https://arxiv.org/abs/2202.10419) by Yin and Neubig (2022) proposes a contrastive attribution method that can be used to answer this question. The method is integrated in Inseq and can be used as follows:

In [7]:
import inseq

model = inseq.load_model("google/flan-t5-base", "input_x_gradient")

out = model.attribute(
    "Does 3 + 3 equal 6?",
    # Fix the original target
    "yes",
    attributed_fn="contrast_prob_diff",
    # Also show the probability delta between the two options
    step_scores=["contrast_prob_diff", "probability"],
    contrast_targets="no",
)

# Normally attributions are normalized to sum up to 1
# Here we want to see how they contribute to the probability difference
out.show()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Attributing with input_x_gradient...: 100%|██████████| 3/3 [00:02<00:00,  1.16s/it]


,,0,1
,,▁no → ▁yes,</s>
0,▁Does,0.149,0.0
1,▁3,0.09,0.0
2,▁+,0.089,0.0
3,▁3,0.097,0.0
4,▁equal,0.191,0.0
5,▁6,0.153,0.0
6,?,0.11,0.0
7,</s>,0.121,0.0
,contrast_prob_diff,0.0,-0.0


We can see that the model is relying more heavily on formulation keywords (`Does`, `equal`) and less on the numbers to determine the answer. The gap between the positive and negative answer is also quite small (5%), suggesting that the model is not very confident in its answer. Changing the input to `Does 3 + 3 equal 7?` confirms that the actual expression is not playing a relevant role in the generation.

> ⚠️ **Important**: Since contrastive attribution compares the probabilities of a pair of `(original, contrastive)` tokens, in order for it to work, the compared sequences must have the same length. For example, if "yes" was tokenized as `_y`, `es` we couldn't have compared it with a single token `_no` using Inseq.

### Your turn to attribute

Using the generation model and a task of your choice, try to use contrastive attribution on at least three examples to highlight some interesting pattern of your choice. We encourage you to explore whathever you find most interesting, but here are some suggestions:

- Is negation relevant in producing the correct answer in open question answering models like the one we used in the previous example?

- When translating sentences like `The nurse went to the hospital` to a gendered language like Spanish, Italian or German, the model will have to select a gender for the subject. What is the model relying on to make this choice?

- Considering a fixed example like `Does 3 + 3 = 6?` but using models with increasingly more parameters (e.g. `flan-t5-small`, `flan-t5-base`, `flan-t5-large`), how does input importance and model confidence change?

After producing the visualizations, comment on the results and try to explain what you observe.

Inseq also supports attribution of quantized models (see [example](https://inseq.readthedocs.io/examples/locate_gpt2_knowledge.html)), in case you want to explore using larger models for your analysis. Refer to the [Inseq documentation](https://inseq.readthedocs.io/) for more details.

In [10]:
# Example 1: Negation in factual questions

out = model.attribute(
    "Is the sky not blue during a clear day?",
    "no",
    attributed_fn="contrast_prob_diff",
    step_scores=["contrast_prob_diff", "probability"],
    contrast_targets="yes",
)

out.show()

Attributing with input_x_gradient...: 100%|██████████| 3/3 [00:02<00:00,  1.08s/it]


,,0,1
,,▁yes → ▁no,</s>
0,▁I,0.09,0.0
1,s,0.099,0.0
2,▁the,0.024,0.0
3,▁sky,0.086,0.0
4,▁not,0.072,0.0
5,▁blue,0.087,0.0
6,▁during,0.079,0.0
7,▁,0.022,0.0
8,a,0.024,0.0


In [11]:
# Example 2: Translation to a gendered language

out = model.attribute(
    "Translate to Italian: The nurse went to the hospital.",
    "L'infermiera è andata all'ospedale.",
    attributed_fn="contrast_prob_diff",
    step_scores=["contrast_prob_diff", "probability"],
    contrast_targets="L'infermiere è andato all'ospedale."
)

out.show()

Attributing with input_x_gradient...: 100%|██████████| 21/21 [00:25<00:00,  1.27s/it]


,,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
,,</s> → ▁L,▁L → ',' → in,in → fer,fer → m,mie → ier,re → a,▁,è,▁and,a → at,to → a,▁all,',o,s,pe,dale,.,</s>
0,▁Translat,0.087,0.069,0.051,0.052,0.076,0.08,0.072,0.0,0.0,0.0,0.087,0.097,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,e,0.032,0.022,0.016,0.023,0.022,0.024,0.03,0.0,0.0,0.0,0.029,0.041,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,▁to,0.045,0.042,0.024,0.031,0.045,0.042,0.04,0.0,0.0,0.0,0.044,0.039,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,▁Italian,0.145,0.121,0.073,0.074,0.122,0.114,0.132,0.0,0.0,0.0,0.152,0.129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,:,0.059,0.037,0.041,0.038,0.032,0.032,0.048,0.0,0.0,0.0,0.038,0.043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,▁The,0.102,0.039,0.048,0.042,0.036,0.044,0.073,0.0,0.0,0.0,0.052,0.058,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,▁nurse,0.196,0.428,0.492,0.449,0.384,0.426,0.263,0.0,0.0,0.0,0.114,0.181,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,▁went,0.084,0.041,0.047,0.041,0.04,0.042,0.089,0.0,0.0,0.0,0.175,0.112,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,▁to,0.03,0.023,0.017,0.024,0.026,0.017,0.034,0.0,0.0,0.0,0.08,0.048,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# Example 3: Reasoning

models = [ # Trying different models to make a comparison in their behavior
    "google/flan-t5-small",
    "google/flan-t5-base",
    "google/flan-t5-large"
]

prompt = "Does 3 + 3 = 6?"

for m in models:

    model = inseq.load_model(m, "input_x_gradient")

    out = model.attribute(
        prompt,
        "yes",
        attributed_fn="contrast_prob_diff",
        step_scores=["contrast_prob_diff", "probability"],
        contrast_targets="no",
    )

    print("MODEL:", m)
    out.show()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Attributing with input_x_gradient...: 100%|██████████| 3/3 [00:06<00:00,  3.08s/it]

MODEL: google/flan-t5-large


,,0,1
,,▁no → ▁yes,</s>
0,▁Does,0.14,0.0
1,▁3,0.12,0.0
2,▁+,0.136,0.0
3,▁3,0.137,0.0
4,▁=,0.124,0.0
5,▁6,0.083,0.0
6,?,0.088,0.0
7,</s>,0.174,0.0
,contrast_prob_diff,-0.0,0.0
